# 4.3 Sentence Segmentation


## 📝 Finding Sentence Boundaries

By default, spaCy uses the Dependency Parser to figure out where sentences begin and end. This is highly accurate because it understands the grammar of the sentence, not just looking for periods.

You access sentences via `doc.sents` (which returns a generator of `Span` objects).


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

text = "Here's a sentence. Dr. Smith is here! He brought apples, oranges, etc. And we are happy."
doc = nlp(text)

print("Default Dependency Parser Segmentation:")
for i, sent in enumerate(doc.sents):
    print(f"[{i+1}] {sent.text}")


## ⚡ The Sentencizer (Fast, Rule-Based)

If you are only doing text extraction and don't care about dependencies, running the full parser is too slow.
Instead, you can use the **`sentencizer`** pipeline component. It uses simple punctuation rules (like splitting on `.` or `!`).


In [ ]:
nlp_fast = spacy.blank("en") # Create an empty pipeline
nlp_fast.add_pipe("sentencizer") # Add only the sentencizer

doc_fast = nlp_fast(text)
print("Rule-based Sentencizer Segmentation:")
for i, sent in enumerate(doc_fast.sents):
    print(f"[{i+1}] {sent.text}")


Notice that the `sentencizer` might make mistakes with things like "Dr. Smith" or "etc." if not configured properly, whereas the Dependency Parser is much smarter about them!


## 🛠️ Custom Sentence Boundaries

What if your text is split by newlines `\n` instead of periods? You can write a custom component to force sentence boundaries before the parser runs by modifying `token.is_sent_start`.


In [ ]:
from spacy.language import Language

@Language.component("custom_newline_boundaries")
def set_custom_boundaries(doc):
    for token in doc[:-1]:
        if token.text == "\n":
            doc[token.i + 1].is_sent_start = True
    return doc

# Add our custom component before the parser
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("custom_newline_boundaries", before="parser")

text = "First line\nSecond line without period\nThird line"
doc = nlp(text)

print("Custom Boundaries:")
for sent in doc.sents:
    print(f"- {sent.text.strip()}")
